## Ingest data from API football 

In [0]:
import json
import os
import time
from datetime import date, datetime, timezone

import requests

In [0]:
dbutils.widgets.dropdown("mode", "historical", ["historical", "live"], "Mode")
MODE = dbutils.widgets.get("mode")

RAW_PATH = "/Volumes/football_scouting/default/raw"
BASE_URL = "https://v3.football.api-sports.io"


# Leagues to scout - ordered by priority (loaded from top to bottom).
LEAGUES = {
    # Europe - top 5
    39: "Premier League (England)",
    140: "La Liga (Spain)",
    135: "Serie A (Italy)",
    78: "Bundesliga (Germany)",
    61: "Ligue 1 (France)",
    # Europe - other interesting scouting markets
    88: "Eredivisie (Netherlands)",
    94: "Primeira Liga (Portugal)",
    144: "Jupiler Pro League (Belgium)",
    40: "Championship (England)",
    203: "Super Lig (Turkey)",
    179: "Premiership (Scotland)",
    218: "Bundesliga (Austria)",
    207: "Super League (Switzerland)",
    119: "Superliga (Denmark)",
    113: "Allsvenskan (Sweden)",
    103: "Eliteserien (Norway)",
    # South America
    71: "Serie A (Brazil)",
    128: "Liga Profesional (Argentina)",
    # North America
    253: "Major League Soccer (USA)",
    262: "Liga MX (Mexico)",
}

# Load every season from the year 2022 up to the current one.
FIRST_SEASON = 2022

MAX_CALLS_PER_RUN = 6000
SECONDS_BETWEEN_CALLS = 0.25

print(f"Mode: {MODE} | {len(LEAGUES)} leagues | seasons from {FIRST_SEASON} | max {MAX_CALLS_PER_RUN} calls")




##  API key (secret) - reads the key from the Databricks secret scope `football`

In [0]:
# API key (secret)  reads the key from the Databricks secret scope `football`

try:
    API_KEY = dbutils.secrets.get(scope="football", key="api_football_key")
    print("API key loaded from secret scope 'football'")
except Exception:
    dbutils.widgets.text("api_key", "", "API key (only if no secret)")
    API_KEY = dbutils.widgets.get("api_key")

if not API_KEY:
    raise ValueError("No API key found. Create the secret or paste the key in the 'api_key' box.")

## Calling the API

In [0]:
class BudgetUsedUp(Exception):
    """We used all API calls we allow ourselves in this run (or the daily limit is reached)."""


class SeasonNotInPlan(Exception):
    """Our API plan doesn't include this league/season."""


calls_made = 0

def call_api(endpoint, params=None):
    """Call one API-Football endpoint and return the JSON answer (a dict)."""
    global calls_made

    while True:
        if calls_made >= MAX_CALLS_PER_RUN:
            raise BudgetUsedUp()

        time.sleep(SECONDS_BETWEEN_CALLS)  # stay under the "calls per minute" limit
        response = requests.get(
            f"{BASE_URL}/{endpoint}",
            headers={"x-apisports-key": API_KEY},
            params=params,
            timeout=60,
        )
        calls_made += 1
        response.raise_for_status()
        body = response.json()

        # API-Football reports problems in the "errors" field (the HTTP status is still 200)
        errors = body.get("errors")
        if not errors:
            return body

        message = json.dumps(errors)
        if "rateLimit" in message:
            print("Too many calls per minute - waiting 60 seconds...")
            time.sleep(60)
        elif "plan" in message:
            raise SeasonNotInPlan(message)
        elif "requests" in message:
            print(f"Daily limit reached: {message}")
            raise BudgetUsedUp()
        else:
            raise Exception(f"API error for {endpoint} {params}: {message}")


def call_api_all_pages(endpoint, params):
    """Some endpoints (like players) returns 20 records per page. Get all pages."""
    records = []
    page = 1
    while True:
        body = call_api(endpoint, {**params, "page": page})
        records += body["response"]
        if page >= body["paging"]["total"]:
            return records
        page += 1